In [10]:
%matplotlib inline
# Cell 1 — Load the WO-02 payloads (fixtures + population). Read-only; the API is never called.
import json, collections
from pathlib import Path
import pandas as pd
import scripts.shared.db_utils as db_utils

ROOT  = Path(db_utils.__file__).parent.parent.parent
OUT   = ROOT / 'output' / 'edop' / 'dashboard' / 'wo02'
STEP2 = OUT / 'step2'
STEP2.mkdir(parents=True, exist_ok=True)

def load_set(name):
    man = json.loads((OUT / f'manifest_{name}.json').read_text())
    items = []
    for e in man['entries']:
        r = e['result']
        body = json.loads((OUT / r['file']).read_text())
        items.append(dict(id=e['id'], set=name, params=e['params'], http=r['http'],
                          bytes=r['bytes'], seconds=r['seconds'], body=body))
    return man, items

man_f, fx  = load_set('fixtures')
man_p, pop = load_set('population')
ALL = fx + pop
Q   = [q for q in ALL if q['http'] == 200]
ERR = [q for q in ALL if q['http'] != 200]

print('provenance (fixtures):', man_f['provenance'])
print('provenance (population):', man_p['provenance'])
print(f'queries: {len(ALL)} | 200: {len(Q)} | errors: {[(q["id"], q["http"]) for q in ERR]}')
print('by scope/level:', dict(collections.Counter((q['body']['scope']['type'], q['params']['level']) for q in Q)))


provenance (fixtures): {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
provenance (population): {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
queries: 540 | 200: 538 | errors: [('F31', 404), ('P156b', 404)]
by scope/level: {('single_basin', 6): 326, ('single_basin', 8): 103, ('buffer', 6): 104, ('polity', 6): 4, ('polity', 8): 1}


In [11]:
# Cell 2 — Flatten every row of every payload into one list (R); nothing is edited
CS_FIELDS = ['variable', 'friendly_name', 'db_col', 'band', 'method', 'unit_type', 'n_units',
             'representative_score', 'representative_raw', 'units', 'score_suppressed', 'status',
             'coverage', 'coherence', 'modality', 'weight_at_zero', 'caveat', 'year', 'epoch_year', 'detail']

R = []
for q in Q:
    b  = q['body']
    sc = b['scope']['type']
    for row in b['rows']:
        det  = row.get('detail') if isinstance(row.get('detail'), dict) else {}
        dist = det.get('distribution') if isinstance(det.get('distribution'), dict) else {}
        R.append(dict(qid=q['id'], set=q['set'], scope=sc, level=q['params']['level'],
                      grp='T' if row.get('band') == 'T' else 'A-E',
                      variable=row.get('variable'), method=row.get('method'), status=row.get('status'),
                      keys=frozenset(row), nonnull=frozenset(k for k, v in row.items() if v is not None),
                      det_keys=frozenset(det), dist_keys=frozenset(dist), row=row))

def state(rows, f):
    """all = present and non-null in every row; null = present but always null; absent = key never there; n/N = non-null count"""
    n  = len(rows)
    p  = sum(1 for r in rows if f in r['keys'])
    nn = sum(1 for r in rows if f in r['nonnull'])
    if p == 0: return 'absent'
    if nn == n: return 'all'
    if nn == 0: return 'null'
    return f'{nn}/{n}'

print('rows flattened:', len(R))
print(pd.Series(collections.Counter((r['scope'], r['level'], r['grp']) for r in R)).to_string())


rows flattened: 29007
single_basin  6  A-E    16952
              8  A-E     5356
buffer        6  A-E     5408
polity        6  A-E      156
              8  A-E       52
single_basin  6  T        870
polity        6  T        213


In [4]:
# Cell 3 — Matrix A: row fields (cheat sheet 2.3) by scope x level x method x status, Bands A-E
MATRIX = []
for (sc, lv), _ in sorted(collections.Counter((r['scope'], r['level']) for r in R).items()):
    sub = [r for r in R if r['scope'] == sc and r['level'] == lv and r['grp'] == 'A-E']
    if not sub:
        continue
    groups = collections.defaultdict(list)
    for r in sub:
        groups[(r['method'], r['status'])].append(r)
    table = {}
    for (m, s), rs in sorted(groups.items(), key=lambda kv: str(kv[0])):
        label = f'{m} | {s} (n={len(rs)})'
        table[label] = {f: state(rs, f) for f in CS_FIELDS}
        for f in CS_FIELDS:
            MATRIX.append(dict(scope=sc, level=lv, grp='A-E', method=m, status=s, field=f, state=table[label][f], n_rows=len(rs)))
    print(f'=== {sc} L{lv}: {len(sub)} rows')
    print(pd.DataFrame(table).T.to_string())
    print()


=== buffer L6: 5408 rows
                                              variable friendly_name   db_col band method unit_type n_units representative_score representative_raw    units score_suppressed status coverage  coherence modality weight_at_zero caveat  year epoch_year detail
area_weighted | no_data (n=108)                    all           all      all  all    all       all     all                 null               null     null              all    all      all       null      all           null    all  null       null    all
area_weighted | ok (n=3092)                        all           all      all  all    all       all     all            2339/3092            35/3092  33/3092              all    all      all  2805/3092      all            all    all  null       null    all
area_weighted | outside_active_domain (n=440)      all           all      all  all    all       all     all                 null               null     null              all    all      all       null      a

In [5]:
# Cell 4 — Matrix B: row fields for Band T rows, by scope x source family x status
for (sc, lv), _ in sorted(collections.Counter((r['scope'], r['level']) for r in R).items()):
    sub = [r for r in R if r['scope'] == sc and r['level'] == lv and r['grp'] == 'T']
    if not sub:
        continue
    groups = collections.defaultdict(list)
    for r in sub:
        groups[((r['variable'] or '').split('_')[0], r['method'], r['status'])].append(r)
    table = {}
    for (fam, m, s), rs in sorted(groups.items(), key=lambda kv: str(kv[0])):
        label = f'{fam} | {m} | {s} (n={len(rs)})'
        table[label] = {f: state(rs, f) for f in CS_FIELDS}
        for f in CS_FIELDS:
            MATRIX.append(dict(scope=sc, level=lv, grp='T', method=f'{fam}/{m}', status=s, field=f, state=table[label][f], n_rows=len(rs)))
    print(f'=== {sc} L{lv}: {len(sub)} Band T rows')
    print(pd.DataFrame(table).T.to_string())
    print()


=== polity L6: 213 Band T rows
                                           variable friendly_name  db_col band method unit_type n_units representative_score representative_raw units score_suppressed status coverage coherence modality weight_at_zero caveat year epoch_year detail
evolv2k | global_forcing | ok (n=8)             all        absent  absent  all    all       all     all                 null                all   all              all    all     null      null     null           null    all  all       null    all
hyde | grid_areal_distribution | ok (n=4)       all        absent  absent  all    all       all     all                 null                all   all              all    all      all      null     null           null    all  all        all    all
lmr | grid_areal_distribution | ok (n=201)      all        absent  absent  all    all       all     all                 null                all   all              all    all      all      null     null           null    all  all

In [6]:
# Cell 5 — Discrepancies against cheat sheet 2.3 / 2.8: fields the doc names that no payload has, and fields payloads carry that the doc does not name
DOC_TOP     = {'scope', 'shortfall', 'bands', 'temporal', 'caveats', 'rows'}
DOC_CAVEATS = {'lmr_caveat', 'hyde_caveat', 'hyde_nearest_year'}

top_seen = collections.Counter(k for q in Q for k in q['body'])
print('top-level keys seen:', dict(top_seen))
print('  documented but never seen:', sorted(DOC_TOP - set(top_seen)))
print('  seen but not documented  :', sorted(set(top_seen) - DOC_TOP))
print()

row_seen = collections.Counter(k for r in R for k in r['keys'])
print('row keys never seen, though documented:', [f for f in CS_FIELDS if f not in row_seen])
extra = {k: c for k, c in row_seen.items() if k not in CS_FIELDS}
print('row keys seen, not in the documented list:')
for k, c in sorted(extra.items()):
    ex = next(r for r in R if k in r['keys'])
    print(f'  {k}: {c} rows, e.g. {ex["variable"]} ({ex["method"]}, {ex["scope"]})')
print()

cav_top = collections.Counter(k for q in Q for k in (q['body'].get('caveats') or {}))
cav_row = collections.Counter(c for r in R for c in (r['row'].get('caveat') or []))
print('top-level caveats keys:', dict(cav_top), '| documented but not seen:', sorted(DOC_CAVEATS - set(cav_top)), '| seen, undocumented:', sorted(set(cav_top) - DOC_CAVEATS))
print('row-level caveat values:', dict(cav_row))
print()

print('scope block keys by scope type (count of queries):')
sk = collections.defaultdict(collections.Counter)
nq = collections.Counter(q['body']['scope']['type'] for q in Q)
for q in Q:
    for k in q['body']['scope']:
        sk[q['body']['scope']['type']][k] += 1
print(pd.DataFrame({sc: {k: f'{c}/{nq[sc]}' for k, c in cnt.items()} for sc, cnt in sk.items()}).fillna('-').to_string())


top-level keys seen: {'scope': 537, 'shortfall': 537, 'bands': 537, 'temporal': 537, 'caveats': 537, 'rows': 537, 'modality_post_pass': 5, 'member_ids': 5, 'resolver': 5, 'band_t_span': 1}
  documented but never seen: []
  seen but not documented  : ['band_t_span', 'member_ids', 'modality_post_pass', 'resolver']

row keys never seen, though documented: []
row keys seen, not in the documented list:

top-level caveats keys: {'lmr_caveat': 5, 'hyde_nearest_year': 2} | documented but not seen: ['hyde_caveat'] | seen, undocumented: []
row-level caveat values: {'lmr_caveat': 903, 'hyde_nearest_year': 8}

scope block keys by scope type (count of queries):
                  single_basin   buffer polity
type                   428/428  104/104    5/5
lat                    428/428  104/104      -
lon                    428/428  104/104      -
level                  428/428  104/104    5/5
hybas_id               428/428        -      -
n_units                428/428  104/104    5/5
unit_type     

In [7]:
# Cell 6 — Matrix C: `detail` keys by method (cheat sheet 2.4 lists them), with the documented set alongside
DOC_DETAIL = {
    'area_weighted':     {'core_share', 'window_lo', 'window_hi', 'iqr', 'p10', 'p25', 'p75', 'p90', 'spread', 'unit', 'distribution'},
    'dominant_basin':    {'dominant_hybas_id'},
    'class_mixture':     {'mixture', 'modal_class_id', 'modal_share', 'n_classes', 'concentration'},
    'flag_fraction':     set(),
    'distribution_only': {'core_share', 'window_lo', 'window_hi', 'iqr', 'p10', 'p25', 'p75', 'p90', 'spread', 'unit', 'distribution', 'regimes', 'suppressed_score'},
}
DET = []
for m in sorted({r['method'] for r in R if r['grp'] == 'A-E'}, key=str):
    sub = [r for r in R if r['grp'] == 'A-E' and r['method'] == m]
    cnt = collections.Counter(k for r in sub for k in r['det_keys'])
    n_with = sum(1 for r in sub if r['det_keys'])
    seen = set(cnt)
    doc  = DOC_DETAIL.get(m)
    print(f'--- {m}: {len(sub)} rows, {n_with} with a detail block')
    print('   keys (rows carrying):', dict(sorted(cnt.items())))
    if doc is not None:
        print('   documented, never seen:', sorted(doc - seen), '| seen, not documented:', sorted(seen - doc))
    else:
        print('   method not in the documented list')
    for k, c in cnt.items():
        DET.append(dict(method=m, key=k, rows=c, of=len(sub)))
    # which statuses have no detail block at all
    nod = collections.Counter(r['status'] for r in sub if not r['det_keys'])
    if nod: print('   rows with NO detail block, by status:', dict(nod))
    print()


--- area_weighted: 18760 rows, 17950 with a detail block
   keys (rows carrying): {'core_share': 17950, 'distribution': 17950, 'iqr': 17950, 'p10': 17950, 'p25': 17950, 'p75': 17950, 'p90': 17950, 'regimes': 753, 'spread': 17950, 'suppressed_score': 753, 'unit': 17950, 'window_hi': 17950, 'window_lo': 17950}
   documented, never seen: [] | seen, not documented: ['regimes', 'suppressed_score']
   rows with NO detail block, by status: {'no_data': 810}

--- class_mixture: 5360 rows, 5360 with a detail block
   keys (rows carrying): {'concentration': 5360, 'mixture': 5360, 'modal_class_id': 5360, 'modal_label': 142, 'modal_share': 5360, 'n_classes': 5360}
   documented, never seen: [] | seen, not documented: ['modal_label']

--- distribution_only: 1608 rows, 1608 with a detail block
   keys (rows carrying): {'core_share': 1608, 'iqr': 1608, 'p10': 1608, 'p25': 1608, 'p75': 1608, 'p90': 1608, 'regimes': 75, 'spread': 1608, 'suppressed_score': 75, 'unit': 1608, 'window_hi': 1608, 'window_lo'

In [8]:
# Cell 7 — `detail.distribution` sub-block keys (cheat sheet: bins, weights, min, max, mean, n_units, ...)
DOC_DIST = {'bins', 'weights', 'min', 'max', 'mean', 'n_units'}
for m in sorted({r['method'] for r in R if r['dist_keys']}, key=str):
    sub = [r for r in R if r['method'] == m and r['dist_keys']]
    cnt = collections.Counter(k for r in sub for k in r['dist_keys'])
    print(f'{m}: {len(sub)} rows with a distribution block; keys {dict(sorted(cnt.items()))}')
    print('   documented, never seen:', sorted(DOC_DIST - set(cnt)))
ex = next((r for r in R if r['dist_keys'] and r['scope'] == 'buffer'), None)
if ex:
    d = ex['row']['detail']['distribution']
    print('example (buffer):', ex['qid'], ex['variable'], {k: (type(v).__name__, len(v) if hasattr(v, '__len__') else v) for k, v in d.items()})


area_weighted: 17950 rows with a distribution block; keys {'band_t_from': 17950, 'band_t_to': 17950, 'bins': 17950, 'low_resolution': 17950, 'max': 17950, 'mean': 17950, 'min': 17950, 'n_units': 17950, 'p10': 17950, 'p90': 17950, 'resolver_year': 17950, 'unit_type': 17950, 'weights': 17950}
   documented, never seen: []
grid_areal_distribution: 931 rows with a distribution block; keys {'band_t_from': 931, 'band_t_to': 931, 'bins': 931, 'low_resolution': 931, 'max': 931, 'mean': 931, 'min': 931, 'n_units': 931, 'p10': 931, 'p90': 931, 'resolver_year': 931, 'unit_type': 931, 'weights': 931}
   documented, never seen: []
example (buffer): F03 aridity {'bins': ('list', 21), 'weights': ('list', 20), 'n_units': ('int', 4), 'unit_type': ('str', 5), 'low_resolution': ('bool', False), 'min': ('float', 8.3008), 'max': ('float', 13.8204), 'p10': ('float', 8.3008), 'p90': ('float', 10.8197), 'mean': ('float', 9.9625), 'resolver_year': ('NoneType', None), 'band_t_from': ('NoneType', None), 'band_t_

In [12]:
# Cell 8 — Band T specifics: what each source carries, and whether `low_resolution` (named in the WO) exists in any payload
T = [r for r in R if r['grp'] == 'T']
fam = lambda r: (r['variable'] or '').split('_')[0]
for f_ in sorted({fam(r) for r in T}):
    sub = [r for r in T if fam(r) == f_]
    print(f'--- {f_}: {len(sub)} rows in {len({r["qid"] for r in sub})} queries; methods {dict(collections.Counter(r["method"] for r in sub))}; unit_type {dict(collections.Counter(r["row"].get("unit_type") for r in sub))}')
    print('   detail keys:', dict(sorted(collections.Counter(k for r in sub for k in r["det_keys"]).items())))
    print('   caveat values:', dict(collections.Counter(c for r in sub for c in (r["row"].get("caveat") or []))))
    print('   year / epoch_year non-null rows:', sum(1 for r in sub if r["row"].get("year") is not None), '/', sum(1 for r in sub if r["row"].get("epoch_year") is not None))

def mentions(term):
    hits = collections.Counter()
    for r in R:
        for k in r['keys'] | r['det_keys'] | r['dist_keys']:
            if term in k: hits[('key', k)] += 1
        for c in (r['row'].get('caveat') or []):
            if term in str(c): hits[('caveat', c)] += 1
    for q in Q:
        for k, v in (q['body'].get('caveats') or {}).items():
            if term in k or term in str(v): hits[('top caveats', k)] += 1
    return dict(hits)
print()
print('low_resolution anywhere:', mentions('low_res'))
print('HYDE nearest-year info by query:')
for q in Q:
    cav = q['body'].get('caveats') or {}
    if 'hyde_nearest_year' in cav or q['body'].get('temporal'):
        print('  ', q['id'], q['body'].get('temporal'), {k: cav[k] for k in cav if k.startswith('hyde')})


--- evolv2k: 41 rows in 5 queries; methods {'global_forcing': 41}; unit_type {'global': 41}
   detail keys: {}
   caveat values: {}
   year / epoch_year non-null rows: 41 / 0
--- hyde: 76 rows in 7 queries; methods {'grid_areal_distribution': 76}; unit_type {'hyde_cell': 76}
   detail keys: {'distribution': 76, 'p10': 76, 'p90': 76, 'sd': 76, 'unit': 76, 'w_eff': 76}
   caveat values: {'hyde_nearest_year': 8, 'hyde_caveat': 4}
   year / epoch_year non-null rows: 76 / 76
--- lmr: 966 rows in 6 queries; methods {'grid_areal_distribution': 966}; unit_type {'lmr_cell': 966}
   detail keys: {'distribution': 966}
   caveat values: {'lmr_caveat': 966}
   year / epoch_year non-null rows: 966 / 0

low_resolution anywhere: {('key', 'low_resolution'): 19027}
HYDE nearest-year info by query:
   F26 {'from_year': 1475, 'to_year': 1525} {}
   F27 {'from_year': 1250, 'to_year': 1300} {}
   F28 {'from_year': -200, 'to_year': -100} {'hyde_nearest_year': 'No HYDE 3.4 epoch falls within this span; neares

In [13]:
# Cell 9 — Reconcile the cheat sheet 2.6 counts (Timbuktu single basin F01 and 100 km buffer F04) against the payloads
def combo(qid):
    q = next(q for q in Q if q['id'] == qid)
    c = collections.Counter()
    for row in q['body']['rows']:
        if row['band'] == 'T':
            continue
        c[(row['method'], row['status'], row['representative_score'] is not None, bool(row['score_suppressed']))] += 1
    return c

EXPECT = {
    'F01': {('area_weighted', 'ok', True, False): 31, ('area_weighted', 'outside_active_domain', False, False): 4,
            ('distribution_only', 'ok', True, False): 3, ('dominant_basin', 'ok', True, False): 3,
            ('class_mixture', 'ok', False, False): 10, ('flag_fraction', 'ok', False, False): 1},
    'F04': {('area_weighted', 'ok', True, False): 20, ('area_weighted', 'ok', False, True): 12,
            ('area_weighted', 'outside_active_domain', False, False): 3,
            ('distribution_only', 'ok', True, False): 2, ('distribution_only', 'ok', False, True): 1,
            ('dominant_basin', 'ok', True, False): 3, ('class_mixture', 'ok', False, False): 10,
            ('flag_fraction', 'ok', False, False): 1},
}
for qid, exp in EXPECT.items():
    got = combo(qid)
    print(f'--- {qid}: payload vs cheat sheet 2.6  (method, status, has_score, suppressed)')
    for k in sorted(set(exp) | set(got), key=str):
        flag = '' if exp.get(k, 0) == got.get(k, 0) else '   <-- DIFFERS'
        print(f'   {str(k):75s} doc {exp.get(k, 0):3d}  payload {got.get(k, 0):3d}{flag}')
    print('   totals: doc', sum(exp.values()), 'payload', sum(got.values()))
f04 = next(q for q in Q if q['id'] == 'F04')
print()
print('F04 modality values:', dict(collections.Counter(r['modality'] for r in f04['body']['rows'] if r['band'] != 'T')))
print('F04 suppressed rows:', sum(1 for r in f04['body']['rows'] if r['score_suppressed']))


--- F01: payload vs cheat sheet 2.6  (method, status, has_score, suppressed)
   ('area_weighted', 'ok', True, False)                                        doc  31  payload  31
   ('area_weighted', 'outside_active_domain', False, False)                    doc   4  payload   4
   ('class_mixture', 'ok', False, False)                                       doc  10  payload  10
   ('distribution_only', 'ok', True, False)                                    doc   3  payload   3
   ('dominant_basin', 'ok', True, False)                                       doc   3  payload   3
   ('flag_fraction', 'ok', False, False)                                       doc   1  payload   1
   totals: doc 52 payload 52
--- F04: payload vs cheat sheet 2.6  (method, status, has_score, suppressed)
   ('area_weighted', 'ok', False, True)                                        doc  12  payload  12
   ('area_weighted', 'ok', True, False)                                        doc  20  payload  20
   ('area_weighte

In [14]:
# Cell 10 — Zero-aware rule (WO constraint 6): can a rule see from the payload that a variable is zero-inflated?
ae = [r for r in R if r['grp'] == 'A-E']
wz = collections.defaultdict(lambda: [0, 0])
for r in ae:
    w = r['row'].get('weight_at_zero')
    wz[r['variable']][1] += 1
    if w is not None: wz[r['variable']][0] += 1
have = {v: c for v, c in wz.items() if c[0] > 0}
print(f'variables with a non-null weight_at_zero in at least one row: {len(have)} of {len(wz)}')
print(pd.DataFrame({v: dict(nonnull=c[0], rows=c[1]) for v, c in sorted(have.items())}).T.to_string())
print()
print('any key naming "zero" (row, detail, distribution, top, scope):')
hits = collections.Counter()
for r in R:
    for k in r['keys'] | r['det_keys'] | r['dist_keys']:
        if 'zero' in k: hits[k] += 1
for q in Q:
    for k in list(q['body']) + list(q['body']['scope']):
        if 'zero' in k: hits[('top/scope', k)] += 1
print(dict(hits))
print()
print('weight_at_zero range (non-null):', min((r['row']['weight_at_zero'] for r in ae if r['row'].get('weight_at_zero') is not None), default=None),
      '-', max((r['row']['weight_at_zero'] for r in ae if r['row'].get('weight_at_zero') is not None), default=None))
print('score == 0 rows (candidates for the zero-aware branch):', sum(1 for r in ae if r['row'].get('representative_score') == 0))


variables with a non-null weight_at_zero in at least one row: 35 of 52
                             nonnull  rows
aridity                          537   537
aridity_upstream                 537   537
cropland_extent                  537   537
cropland_extent_upstream         537   537
dist_sink                        537   537
elev_max                         537   537
elev_min                         537   537
gw_table_depth                   537   537
human_footprint_09               537   537
human_footprint_09_upstream      537   537
karst                            537   537
karst_upstream                   537   537
pasture_extent                   537   537
pasture_extent_upstream          537   537
pct_clay                         454   537
pct_clay_upstream                454   537
pct_sand                         454   537
pct_sand_upstream                454   537
pct_silt                         454   537
pct_silt_upstream                454   537
permafrost_extent         

In [15]:
# Cell 11 — Payload size, time and row counts by scope x level x bands (cheat sheet section 0 lists payload size as unverified)
rows_ = []
for q in Q:
    b = q['body']
    rows_.append(dict(scope=b['scope']['type'], level=q['params']['level'], bands=q['params']['bands'],
                      kb=q['bytes'] / 1024, secs=q['seconds'], n_rows=len(b['rows'])))
df = pd.DataFrame(rows_)
g = df.groupby(['scope', 'level', 'bands']).agg(n=('kb', 'size'), kb_min=('kb', 'min'), kb_median=('kb', 'median'), kb_max=('kb', 'max'),
                                                  secs_median=('secs', 'median'), secs_max=('secs', 'max'),
                                                  rows_min=('n_rows', 'min'), rows_max=('n_rows', 'max')).round(1)
print(g.to_string())
print()
print('rows per A-E query (distribution):', dict(collections.Counter(len([r for r in q['body']['rows'] if r['band'] != 'T']) for q in Q if q['body']['rows'] and q['params']['bands'] != 'T')))


                             n  kb_min  kb_median  kb_max  secs_median  secs_max  rows_min  rows_max
scope        level bands                                                                            
buffer       6     ABCDE   104    37.7       46.0    48.1          1.2       1.5        52        52
polity       6     ABCDE     3    63.5       69.7    84.3          1.6       1.8        52        52
                   T         1   201.6      201.6   201.6          1.7       1.7       213       213
             8     ABCDE     1    64.7       64.7    64.7          1.4       1.4        52        52
single_basin 6     ABCDE   320    36.3       42.7    43.4          1.2      13.2        52        52
                   ABCDET    6    50.8      159.0   308.7          1.2       1.2        68       372
             8     ABCDE   103    36.4       42.6    43.2          1.0      14.4        52        52

rows per A-E query (distribution): {52: 537}


In [16]:
# Cell 12 — Save the matrices for the results file and the master table
pd.DataFrame(MATRIX).to_csv(STEP2 / 'matrix_row_fields.csv', index=False)
pd.DataFrame(DET).to_csv(STEP2 / 'matrix_detail_keys.csv', index=False)
print('wrote', STEP2 / 'matrix_row_fields.csv')
print('wrote', STEP2 / 'matrix_detail_keys.csv')


wrote /Users/karlg/Documents/repos/_edops/output/edop/dashboard/wo02/step2/matrix_row_fields.csv
wrote /Users/karlg/Documents/repos/_edops/output/edop/dashboard/wo02/step2/matrix_detail_keys.csv
